In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt

In [ ]:
base_folder = r"/storage/alplakes_test/lucerne_100m_2025"

In [ ]:
wind_folder = os.path.join(base_folder, 'wind_analysis')

In [ ]:
catalogue_folder = os.path.join(base_folder, "outputs_swirl", "eddy_catalogues_final")

# Import datasets

In [ ]:
ds_wind = xr.open_dataset(os.path.join(wind_folder, "wind_stats_per_zone.nc"))

In [ ]:
lvl1_csv_path = os.path.join(catalogue_folder, "lvl1.csv")
df_lvl1 = pd.read_csv(lvl1_csv_path)
df_lvl1 = df_lvl1.set_index('id', drop=False)
df_lvl1['date'] = pd.to_datetime(df_lvl1['date'])

In [ ]:
df_lvl1['surface_km2'] = df_lvl1['surface_area_mean_[m2]'] / 1e6

# Number of eddies at different depths

In [ ]:
def filter_by_depths(df, depth_min, depth_max):
    depth_filter = (
            (abs(df['depth_min_[m]']) >= abs(depth_max)) &
            (abs(df['depth_max_[m]']) <= abs(depth_min))
    )

    return df[depth_filter]

In [ ]:
depths_range = [(-2, -0), (-5, -2), (-10, -5), (-10, 0)] # [(-2, -0), (-5, -2), (-10, -5), (-30, -10), (-50, -30)]

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="black", zorder=1, add_legend=False, alpha=1, label='Wind')
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")
ax1.legend(loc="upper left")

# --- RIGHT axis : Eddy numbers ---
ax2 = ax1.twinx()
ax2.set_ylabel("Number of Eddies within that layer [-]")
for depth_min, depth_max in depths_range:
    df_lvl1_filtered_by_depth = filter_by_depths(df_lvl1, depth_min, depth_max)
    nb_eddy_by_depth = df_lvl1_filtered_by_depth.groupby(['date'])['id'].count()
    nb_eddy_by_depth.plot(ax=ax2, label = f'{depth_max} to {depth_min}m', alpha=0.9)

ax2.grid(False)
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
ax2.legend(loc="upper right")

xlim = (f"2025-10-20 00:30:00", f"2025-11-01 00:30:00")
ax2.set_xlim(xlim)
fig.tight_layout()

In [ ]:
os.makedirs(os.path.join(wind_folder, 'wind_eddy_number_analysis'), exist_ok=True)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="black", zorder=1, add_legend=False, alpha=1, label='Wind')
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")
ax1.legend(loc="upper left")

# --- RIGHT axis : Eddy numbers ---
ax2 = ax1.twinx()
ax2.set_ylabel("Number of Eddies within that layer [-]")
for depth_min, depth_max in depths_range:
    df_lvl1_filtered_by_depth = filter_by_depths(df_lvl1, depth_min, depth_max)
    nb_eddy_by_depth = df_lvl1_filtered_by_depth.groupby(['date'])['id'].count()
    nb_eddy_by_depth.plot(ax=ax2, label = f'{depth_max} to {depth_min}m', alpha=0.9)

ax2.grid(False)
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
ax2.legend(loc="upper right")

# --- x limits ---
for month in range(1, 12):
    xlim = (f"2025-{month:02}-01 00:30:00", f"2025-{month+1:02}-01 00:30:00")
    ax2.set_xlim(xlim)
    fig.tight_layout()
    fig.savefig(os.path.join(wind_folder, 'wind_eddy_number_analysis', f'month_{month:02}.png'))

# Number of eddies per surface

In [ ]:
df_lvl1_filt = filter_by_depths(df_lvl1, -15, 0).copy()

In [ ]:
surf_range = [(0.01, 5e-1), (5e-1, 1), (1, 10)]

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="black", zorder=1, add_legend=False, alpha=1, label='Wind')
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")
ax1.legend(loc="upper left")

# --- RIGHT axis : Eddy numbers ---
ax2 = ax1.twinx()
ax2.set_ylabel("Number of Eddies with that surface [-]")
for surf_min, surf_max in surf_range:
    df_lvl1_filtered_by_surface = df_lvl1_filt[
        (df_lvl1_filt['surface_km2'] > surf_min) &
        (df_lvl1_filt['surface_km2'] < surf_max)
    ]
    nb_eddy_by_surf = df_lvl1_filtered_by_surface.groupby(['date'])['id'].count()
    nb_eddy_by_surf.plot(ax=ax2, label = f'{surf_min} to {surf_max} km$^2$', alpha=0.9)

ax2.grid(False)
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
ax2.legend(loc="upper right")
plt.title('Number of Eddies in the first 15m')

# --- x limits ---
for month in range(1, 12):
    xlim = (f"2025-{month:02}-01 00:30:00", f"2025-{month+1:02}-01 00:30:00")
    ax2.set_xlim(xlim)
    fig.tight_layout()
    fig.savefig(os.path.join(wind_folder, 'wind_eddy_number_analysis','by_surface', f'by_surface_month_{month:02}.png'))

In [ ]:
fig, ax1 = plt.subplots(figsize=(30, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="black", zorder=1, add_legend=False, alpha=1, label='Wind')
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")
ax1.legend(loc="upper left")

# --- RIGHT axis : Eddy numbers ---
ax2 = ax1.twinx()
ax2.set_ylabel("Number of Eddies with that surface [-]")
for surf_min, surf_max in surf_range:
    df_lvl1_filtered_by_surface = df_lvl1_filt[
        (df_lvl1_filt['surface_km2'] > surf_min) &
        (df_lvl1_filt['surface_km2'] < surf_max)
    ]
    nb_eddy_by_surf = df_lvl1_filtered_by_surface.groupby(['date'])['id'].count()
    nb_eddy_by_surf.plot(ax=ax2, label = f'{surf_min} to {surf_max} km$^2$', alpha=0.5)

ax2.grid(False)
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
ax2.legend(loc="upper right")
plt.title('Number of Eddies in the first 15m')

xlim = (f"2025-01-01 00:30:00", f"2025-12-31 00:30:00")
ax2.set_xlim(xlim)

fig.tight_layout()
fig.savefig(os.path.join(wind_folder, 'wind_eddy_number_analysis','by_surface', f'by_surface_month_all.png'))

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# --- LEFT axis: wind ---
ds_wind.sel(zone="all").speed.plot(ax=ax1, c="black", zorder=1, add_legend=False, alpha=1, label='Wind')
ax1.set_ylabel("Wind Speed (m/s)")
ax1.set_title("")
ax1.legend(loc="upper left")

# --- RIGHT axis : Eddy numbers ---
ax2 = ax1.twinx()
ax2.set_ylabel("Number of Eddies with that surface [-]")
for surf_min, surf_max in surf_range:
    df_lvl1_filtered_by_surface = df_lvl1_filt[
        (df_lvl1_filt['surface_km2'] > surf_min) &
        (df_lvl1_filt['surface_km2'] < surf_max)
    ]
    nb_eddy_by_surf = df_lvl1_filtered_by_surface.groupby(['date'])['id'].count()
    nb_eddy_by_surf.plot(ax=ax2, label = f'{surf_min} to {surf_max} km$^2$', alpha=0.5)

ax2.grid(False)
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
ax2.legend(loc="upper right")
plt.title('Number of Eddies in the first 15m')

xlim = (f"2025-03-13 00:30:00", f"2025-03-16 00:30:00")
ax2.set_xlim(xlim)

# Quantification

In [ ]:
df_lvl1_filt = filter_by_depths(df_lvl1, -10, 0).copy()

In [ ]:
surface_threshold = 0 # km
df_lvl1_filt = df_lvl1_filt[df_lvl1_filt['surface_km2'] > surface_threshold]

In [ ]:
eddy_numbers = (df_lvl1_filt.groupby(['date'])['id']
                     .count().sort_index()
                     .rolling(window=6, center=True)
                     .mean()
                  )

In [ ]:
ds_wind_speed_all = ds_wind.sel(zone="all")

In [ ]:
def get_eddy_perc_diff(ds_wind_speed_all, eddy_numbers, wind_threshold):
    wind_gt_threshold = ds_wind_speed_all.where(ds_wind_speed_all.speed > wind_threshold, drop=True)
    wind_gt_threshold = wind_gt_threshold['time'].assign_coords(
        day=wind_gt_threshold["time"].dt.floor("D")
    )
    wind_gt_threshold_day = wind_gt_threshold.groupby(['day']).first()

    dates = []
    eddy_nb_before = []
    eddy_nb_during = []
    eddy_nb_after = []
    for sel_date in wind_gt_threshold_day.values:
        sel_date = sel_date - pd.offsets.Minute(30)
        dates.append(sel_date)
        eddy_nb_before.append(eddy_numbers.get(sel_date - pd.Timedelta(hours=6), np.nan))
        eddy_nb_during.append(eddy_numbers.get(sel_date, 0))
        eddy_nb_after.append(eddy_numbers.get(sel_date + pd.Timedelta(hours=6), 0))

    df_out = pd.DataFrame({
        "date": pd.to_datetime(dates),
        "eddy_nb_before": eddy_nb_before,
        "eddy_nb_during": eddy_nb_during,
        "eddy_nb_after": eddy_nb_after,
    })

    df_out['perc_diff'] = 100 * (df_out['eddy_nb_after'] - df_out['eddy_nb_before']) / df_out['eddy_nb_before']
    return df_out['perc_diff'].dropna()



In [ ]:
from matplotlib.ticker import MultipleLocator

thresholds = [3, 4, 6, 8, 10]
colors = ['blue', 'purple', 'red', 'orange', 'yellow']

all_data = {}
for wt in thresholds:
    all_data[wt] = get_eddy_perc_diff(ds_wind_speed_all, eddy_numbers, wt)

all_values = np.concatenate([v.values for v in all_data.values()])
bin_edges = np.linspace(np.nanmin(all_values), np.nanmax(all_values), 25)

fig, (ax, ax_mean) = plt.subplots(2, 1, figsize=(10, 6),
                                  gridspec_kw={'height_ratios': [4, 1]},
                                  sharex=True)

n_thresholds = len(thresholds)
bin_width = (bin_edges[1] - bin_edges[0])
bar_width = bin_width / (n_thresholds + 1)

for i, (wt, color) in enumerate(zip(thresholds, colors)):
    data = all_data[wt]
    counts, _ = np.histogram(data, bins=bin_edges)
    percentages = 100 * counts / counts.sum()
    ymean = data.mean()
    sem = data.std() / np.sqrt(len(data))
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    offset = (i - n_thresholds / 2 + 0.5) * bar_width
    ax.bar(bin_centers + offset, percentages, width=bar_width, color=color, alpha=0.7,
           edgecolor='black', linewidth=0.5,
           label=f"Wind > {wt} m/s (avg = {ymean:.2f}%)")

    # Plot mean ± SEM on the lower subplot
    ax_mean.axvline(ymean, color=color)

ax.set_ylabel("Percentage of events [%]")
ax.set_title("Following a strong wind event")
ax.legend(loc="upper right")

ax_mean.set_yticks(range(n_thresholds))
ax_mean.set_yticklabels([f"> {wt} m/s" for wt in thresholds])
ax_mean.set_xlabel("Percentage change in eddy number [%]")
ax_mean.set_ylabel("Threshold")
ax_mean.set_title('Mean')
ax_mean.axvline(0, color='grey', linestyle=':', alpha=0.5)
ax_mean.xaxis.set_major_locator(MultipleLocator(5))

# Force identical x-limits on both axes
xmin = min(ax.get_xlim()[0], ax_mean.get_xlim()[0])
xmax = max(ax.get_xlim()[1], ax_mean.get_xlim()[1])

ax.set_xlim(xmin, xmax)
ax_mean.set_xlim(xmin, xmax)

plt.tight_layout()